In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "BTCUSDT"
    / "BTCUSDT_1m_baseline.csv"
)

df = pd.read_csv(DATA_PATH)

df.head()

,open,high,low,close,volume,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,target,hour,minute,day_of_week
0,93576.00,93610.93,93537.50,93610.93,8.21827,7.689788e+05,2631,3.95157,369757.326529,1,0,0,2
1,93610.93,93652.00,93606.20,93652.00,12.14029,1.136551e+06,1273,4.08887,382791.500172,1,0,1,2
2,93652.00,93702.15,93635.98,93702.15,11.60597,1.087101e+06,1095,5.86840,549682.868570,0,0,2,2
3,93702.14,93702.15,93654.48,93677.98,8.72958,8.177203e+05,1461,2.48203,232486.113080,0,0,3,2
4,93677.98,93677.99,93659.92,93661.20,5.24749,4.915570e+05,988,0.48880,45786.251963,1,0,4,2


In [3]:
X = df.drop(columns="target")
y = df["target"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dropout(0.2),

    Dense(32, activation="relu"),
    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

d:\Subject\CV2026\Market Risk Classification\market-risk-classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [7]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5073 - loss: 0.7022 - val_accuracy: 0.5099 - val_loss: 0.6925
Epoch 2/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5156 - loss: 0.6958 - val_accuracy: 0.5133 - val_loss: 0.6924
Epoch 3/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5246 - loss: 0.6924 - val_accuracy: 0.5174 - val_loss: 0.6920
Epoch 4/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5203 - loss: 0.6920 - val_accuracy: 0.5152 - val_loss: 0.6935
Epoch 5/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5244 - loss: 0.6921 - val_accuracy: 0.5105 - val_loss: 0.6930
Epoch 6/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5291 - loss: 0.6908 - val_accuracy: 0.5105 - val_loss: 0.6949
Epoch 7/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5243 - loss: 0.6919 - val_accuracy: 0.5099 - val_loss: 0.6937
Epoch 8/20
202/202 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.5308 - loss: 0.6913 - val_accuracy: 0.

In [8]:
y_prob = model.predict(X_test)

y_pred = (y_prob > 0.5).astype(int)

126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 496us/step


In [9]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.5184
Precision: 0.4976
Recall   : 0.1609
F1 Score : 0.2432


In [10]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.52      0.85      0.65      2093
           1       0.50      0.16      0.24      1939

    accuracy                           0.52      4032
   macro avg       0.51      0.51      0.44      4032
weighted avg       0.51      0.52      0.45      4032

